<a href="https://colab.research.google.com/github/adenurchalisa/Automatic-Photo-Clustering-System-Optimization-HDBSCAN/blob/claude/review-thesis-research-4x4MQ/notebooks/15_CLIP_Feature_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 15 — CLIP Feature Extraction

**Tujuan:**  
Mengekstrak fitur foto menggunakan **CLIP (Contrastive Language-Image Pre-Training)** dari OpenAI sebagai alternatif dari pendekatan face-based (InsightFace) yang digunakan sebelumnya.

**Perbedaan utama dengan pipeline sebelumnya:**

| Aspek | InsightFace (Lama) | CLIP (Baru) |
|---|---|---|
| Granularitas | Per-wajah | Per-foto |
| Jumlah embedding | 12.715 (dari 2.533 foto) | 2.533 (1 per foto) |
| Dimensi | 512 | 512 (ViT-B/32) / 768 (ViT-L/14) |
| Basis clustering | Identitas orang | Konten / scene / aktivitas |
| Foto tanpa wajah | Diabaikan | Tetap diproses |

**Model:** `ViT-L/14` — model CLIP terbesar yang tersedia, memberikan representasi visual terbaik.


---
## 1. Instalasi & Import

In [ ]:
# Install CLIP (OpenAI)
!pip install git+https://github.com/openai/CLIP.git -q
!pip install pillow-heif -q

print('Install selesai!')

In [ ]:
import os
import numpy as np
import pandas as pd
import pickle
import warnings
from collections import Counter
from tqdm import tqdm

import torch
import clip
from PIL import Image
import pillow_heif

from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

pillow_heif.register_heif_opener()
warnings.filterwarnings('ignore')

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Libraries loaded!')
print(f'  - CLIP version  : {clip.__version__ if hasattr(clip, "__version__") else "installed"}')
print(f'  - Device        : {DEVICE}')
print(f'  - PyTorch       : {torch.__version__}')
print(f'  - HEIC support  : enabled')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

---
## 2. Konfigurasi

In [ ]:
# =====================================================================
# PATH CONFIGURATION  (sama dengan pipeline InsightFace)
# =====================================================================
DATASET_PATH  = '/content/drive/MyDrive/OTW S.KOM/DOKUMENTASI OUTDOOR 2025'
OUTPUT_DIR    = '/content/drive/MyDrive/OTW S.KOM/Embeddings'
OLD_EMB_PATH  = os.path.join(OUTPUT_DIR, 'embeddings_data.pkl')        # InsightFace lama
CLIP_EMB_PATH = os.path.join(OUTPUT_DIR, 'clip_embeddings_data.pkl')   # CLIP baru

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =====================================================================
# CLIP CONFIGURATION
# =====================================================================
CLIP_MODEL    = 'ViT-L/14'   # Best quality; ganti ke 'ViT-B/32' kalau VRAM tidak cukup
BATCH_SIZE    = 64           # Turunkan ke 32 kalau OOM

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.heic', '.heif'}

print('Configuration set!')
print(f'  - Dataset path  : {DATASET_PATH}')
print(f'  - Output dir    : {OUTPUT_DIR}')
print(f'  - CLIP model    : {CLIP_MODEL}')
print(f'  - Batch size    : {BATCH_SIZE}')
print(f'  - Device        : {DEVICE}')

---
## 3. Load Dataset

In [ ]:
def collect_image_paths(root_dir, extensions=IMAGE_EXTENSIONS):
    image_paths = []
    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            if os.path.splitext(filename)[1].lower() in extensions:
                image_paths.append(os.path.join(dirpath, filename))
    image_paths.sort()
    return image_paths


image_paths = collect_image_paths(DATASET_PATH)

print(f'Total foto ditemukan: {len(image_paths)}')
print(f'\nContoh path (5 pertama):')
for p in image_paths[:5]:
    print(f'  - {p}')

In [ ]:
def get_folder_label(path):
    return os.path.basename(os.path.dirname(path))

# Buat mapping: path → folder label
folder_labels = [get_folder_label(p) for p in image_paths]
folder_counts = Counter(folder_labels)

print('DISTRIBUSI FOTO PER FOLDER')
print('=' * 45)
for folder, count in folder_counts.most_common():
    bar = '█' * (count // 30)
    print(f'  {folder:<25} {count:>4}  {bar}')
print('=' * 45)
print(f'  TOTAL                     {len(image_paths):>4}')

---
## 4. Inisialisasi CLIP

In [ ]:
print(f'Available CLIP models: {clip.available_models()}')
print(f'\nLoading {CLIP_MODEL}...')

model, preprocess = clip.load(CLIP_MODEL, device=DEVICE)
model.eval()  # Inference mode

# Info model
total_params = sum(p.numel() for p in model.parameters())
emb_dim = model.visual.output_dim

print(f'\nCLIP model loaded!')
print(f'  - Model         : {CLIP_MODEL}')
print(f'  - Parameters    : {total_params:,}')
print(f'  - Embedding dim : {emb_dim}')
print(f'  - Device        : {DEVICE}')

In [ ]:
# --- Quick test pada 1 gambar ---
test_img = Image.open(image_paths[0]).convert('RGB')
test_tensor = preprocess(test_img).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    test_emb = model.encode_image(test_tensor)
    test_emb = test_emb / test_emb.norm(dim=-1, keepdim=True)  # L2 normalize

print(f'Test embedding berhasil!')
print(f'  - Input image  : {os.path.basename(image_paths[0])}')
print(f'  - Output shape : {test_emb.shape}  → (1, {emb_dim})')
print(f'  - Dtype        : {test_emb.dtype}')
print(f'  - Norm (harus ~1.0): {test_emb.norm().item():.4f}')

---
## 5. Ekstraksi Fitur CLIP

> **Catatan:** CLIP memproses **seluruh foto** (bukan crop wajah), menghasilkan **1 embedding per foto**.  
> Berbeda dengan InsightFace yang menghasilkan 1 embedding per wajah yang terdeteksi.

In [ ]:
def extract_clip_features(image_paths, model, preprocess, device,
                           batch_size=64, save_path=None):
    """
    Ekstrak CLIP image embeddings dari semua foto.
    Returns:
        dict dengan keys: embeddings, metadata, stats
    """
    embeddings_list = []
    metadata_list   = []
    error_images    = []

    stats = {
        'total_images'     : len(image_paths),
        'images_success'   : 0,
        'images_error'     : 0,
        'embedding_dim'    : model.visual.output_dim,
        'model'            : CLIP_MODEL,
        'normalized'       : True,
    }

    # Batch processing untuk efisiensi GPU
    batches = [image_paths[i:i+batch_size] for i in range(0, len(image_paths), batch_size)]

    print(f'Extracting CLIP features from {len(image_paths)} images...')
    print(f'  Batch size : {batch_size}')
    print(f'  Batches    : {len(batches)}')
    print('=' * 60)

    for batch_paths in tqdm(batches, desc='Batch'):
        batch_tensors = []
        valid_paths   = []

        # Load & preprocess per batch
        for img_path in batch_paths:
            try:
                img = Image.open(img_path).convert('RGB')
                tensor = preprocess(img)
                batch_tensors.append(tensor)
                valid_paths.append(img_path)
            except Exception as e:
                error_images.append({'path': img_path, 'error': str(e)})
                stats['images_error'] += 1

        if not batch_tensors:
            continue

        # Stack ke tensor & pindah ke device
        batch_input = torch.stack(batch_tensors).to(device)

        # Encode
        with torch.no_grad():
            features = model.encode_image(batch_input)         # (B, dim)
            features = features / features.norm(dim=-1, keepdim=True)  # L2 normalize
            features_np = features.cpu().numpy().astype(np.float32)

        # Simpan hasil
        for i, img_path in enumerate(valid_paths):
            embeddings_list.append(features_np[i])
            metadata_list.append({
                'image_path'    : img_path,
                'image_filename': os.path.basename(img_path),
                'folder'        : os.path.basename(os.path.dirname(img_path)),
            })
            stats['images_success'] += 1

    # Convert ke array
    embeddings = np.array(embeddings_list, dtype=np.float32)

    # Summary
    print('\n' + '=' * 60)
    print('EXTRACTION SUMMARY')
    print('=' * 60)
    print(f'  Total foto diproses  : {stats["total_images"]}')
    print(f'  Berhasil             : {stats["images_success"]}')
    print(f'  Error                : {stats["images_error"]}')
    print(f'  Embedding shape      : {embeddings.shape}')
    print(f'  Dimensi              : {embeddings.shape[1]}')
    print(f'  Normalized           : L2 (norm ≈ 1.0)')

    results = {
        'embeddings'  : embeddings,
        'metadata'    : metadata_list,
        'stats'       : stats,
        'error_images': error_images,
    }

    if save_path:
        with open(save_path, 'wb') as f:
            pickle.dump(results, f)
        print(f'\nSaved to: {save_path}')

    return results

In [ ]:
# Jalankan ekstraksi
# Estimasi waktu: ~2-4 menit untuk 2533 foto pada T4 GPU
clip_results = extract_clip_features(
    image_paths=image_paths,
    model=model,
    preprocess=preprocess,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    save_path=CLIP_EMB_PATH
)

clip_embeddings = clip_results['embeddings']
clip_metadata   = clip_results['metadata']
clip_stats      = clip_results['stats']

print(f'\nDone! Shape: {clip_embeddings.shape}')

---
## 6. Analisis Embedding CLIP

In [ ]:
# Statistik dasar
norms = np.linalg.norm(clip_embeddings, axis=1)

print('CLIP Embedding Statistics')
print('=' * 40)
print(f'  Shape          : {clip_embeddings.shape}')
print(f'  dtype          : {clip_embeddings.dtype}')
print(f'  Mean norm      : {norms.mean():.4f}  (harus ≈ 1.0)')
print(f'  Std norm       : {norms.std():.6f}')
print(f'  Value range    : [{clip_embeddings.min():.4f}, {clip_embeddings.max():.4f}]')
print(f'  Mean (per-dim) : {clip_embeddings.mean():.4f}')
print(f'  Std  (per-dim) : {clip_embeddings.std():.4f}')

In [ ]:
# Distribusi per folder dalam embeddings
emb_folders = [m['folder'] for m in clip_metadata]
emb_folder_counts = Counter(emb_folders)

fig, ax = plt.subplots(figsize=(9, 4))
folders_sorted = [k for k, _ in emb_folder_counts.most_common()]
counts_sorted  = [emb_folder_counts[k] for k in folders_sorted]
colors = cm.tab10(np.linspace(0, 1, len(folders_sorted)))

bars = ax.bar(folders_sorted, counts_sorted, color=colors, edgecolor='white', linewidth=0.8)
for bar, count in zip(bars, counts_sorted):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(count), ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Distribusi Foto per Folder (CLIP Embeddings)', fontsize=13, fontweight='bold')
ax.set_xlabel('Folder')
ax.set_ylabel('Jumlah Foto')
ax.set_ylim(0, max(counts_sorted) * 1.15)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

print('\nDistribusi:')
for folder, count in emb_folder_counts.most_common():
    print(f'  {folder:<25} {count:>4} foto  ({count/len(clip_metadata)*100:.1f}%)')

---
## 7. Perbandingan CLIP vs InsightFace

In [ ]:
# Load embeddings InsightFace lama
print(f'Loading InsightFace embeddings dari: {OLD_EMB_PATH}')

with open(OLD_EMB_PATH, 'rb') as f:
    old_data = pickle.load(f)

old_embeddings = old_data['embeddings']
old_metadata   = old_data['metadata']
old_stats      = old_data['stats']

print(f'InsightFace embeddings loaded!')
print(f'  Shape: {old_embeddings.shape}')

In [ ]:
# Tabel perbandingan
print('=' * 60)
print(f'  {"ASPEK":<30} {"InsightFace":>12} {"CLIP":>12}')
print('=' * 60)
print(f'  {"Model":<30} {"buffalo_l (R50)":>12} {CLIP_MODEL:>12}')
print(f'  {"Pendekatan":<30} {"Per-wajah":>12} {"Per-foto":>12}')
print(f'  {"Total embedding":<30} {old_embeddings.shape[0]:>12,} {clip_embeddings.shape[0]:>12,}')
print(f'  {"Dimensi":<30} {old_embeddings.shape[1]:>12} {clip_embeddings.shape[1]:>12}')
print(f'  {"Foto tanpa wajah":<30} {old_stats["images_no_face"]:>12} {"0 (semua diproses)":>12}')
print(f'  {"Foto multi-wajah":<30} {old_stats["images_multi_face"]:>12} {"1 emb/foto":>12}')
print(f'  {"Basis kluster":<30} {"Identitas orang":>12} {"Konten scene":>12}')
print('=' * 60)

In [ ]:
# Coverage: berapa foto yang terwakili?
old_covered_paths = set(m['image_path'] for m in old_metadata)
clip_covered_paths = set(m['image_path'] for m in clip_metadata)
all_paths = set(image_paths)

old_missed  = all_paths - old_covered_paths
clip_missed = all_paths - clip_covered_paths

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# InsightFace coverage
covered_old  = len(old_covered_paths)
missed_old   = len(old_missed)
axes[0].pie([covered_old, missed_old],
            labels=[f'Foto dg wajah\n({covered_old})', f'Tanpa wajah\n({missed_old})'],
            colors=['#4C72B0', '#DD8452'],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
axes[0].set_title('InsightFace — Coverage Foto', fontsize=12, fontweight='bold')

# CLIP coverage
covered_clip = len(clip_covered_paths)
missed_clip  = len(clip_missed)
axes[1].pie([covered_clip, missed_clip if missed_clip > 0 else 1],
            labels=[f'Berhasil diproses\n({covered_clip})',
                    f'Error\n({missed_clip if missed_clip > 0 else 0})'],
            colors=['#55A868', '#C44E52'],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
axes[1].set_title('CLIP — Coverage Foto', fontsize=12, fontweight='bold')

plt.suptitle('Perbandingan Coverage Foto', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'InsightFace : {covered_old}/{len(all_paths)} foto ({covered_old/len(all_paths)*100:.1f}%)')
print(f'CLIP        : {covered_clip}/{len(all_paths)} foto ({covered_clip/len(all_paths)*100:.1f}%)')

---
## 8. Visualisasi PCA — CLIP Embeddings

Proyeksi ke 2D menggunakan PCA untuk melihat struktur distribusi embedding CLIP, diwarnai berdasarkan folder asal foto.

In [ ]:
# PCA 2D — CLIP
pca = PCA(n_components=2, random_state=42)
clip_2d = pca.fit_transform(clip_embeddings)

variance_explained = pca.explained_variance_ratio_
print(f'PCA variance explained: PC1={variance_explained[0]*100:.2f}%, PC2={variance_explained[1]*100:.2f}%')
print(f'Total: {sum(variance_explained)*100:.2f}%')

# Plot
unique_folders = list(set(emb_folders))
palette = cm.tab10(np.linspace(0, 1, len(unique_folders)))
folder_color_map = {f: palette[i] for i, f in enumerate(sorted(unique_folders))}

fig, ax = plt.subplots(figsize=(11, 8))

for folder in sorted(unique_folders):
    mask = [i for i, m in enumerate(clip_metadata) if m['folder'] == folder]
    ax.scatter(clip_2d[mask, 0], clip_2d[mask, 1],
               label=f'{folder} (n={len(mask)})',
               color=folder_color_map[folder],
               alpha=0.55, s=18, edgecolors='none')

ax.set_title(f'PCA 2D — CLIP {CLIP_MODEL} Embeddings\n'
             f'(PC1={variance_explained[0]*100:.1f}%, PC2={variance_explained[1]*100:.1f}%)',
             fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({variance_explained[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({variance_explained[1]*100:.1f}% variance)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# PCA 2D — InsightFace (untuk perbandingan visual)
# Ambil folder label dari metadata InsightFace
old_folders = [os.path.basename(os.path.dirname(m['image_path'])) for m in old_metadata]
unique_old_folders = sorted(set(old_folders))

pca_old = PCA(n_components=2, random_state=42)
old_2d  = pca_old.fit_transform(old_embeddings)

var_old = pca_old.explained_variance_ratio_
print(f'InsightFace PCA variance explained: PC1={var_old[0]*100:.2f}%, PC2={var_old[1]*100:.2f}%')

# Side-by-side plot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

palette2 = cm.tab10(np.linspace(0, 1, len(unique_old_folders)))
old_color_map = {f: palette2[i] for i, f in enumerate(unique_old_folders)}

# InsightFace
for folder in unique_old_folders:
    mask = [i for i, f in enumerate(old_folders) if f == folder]
    axes[0].scatter(old_2d[mask, 0], old_2d[mask, 1],
                    label=f'{folder} (n={len(mask)})',
                    color=old_color_map[folder],
                    alpha=0.4, s=8, edgecolors='none')
axes[0].set_title(f'InsightFace (buffalo_l)\nPCA: {var_old[0]*100:.1f}% + {var_old[1]*100:.1f}% = {sum(var_old)*100:.1f}%',
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({var_old[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({var_old[1]*100:.1f}%)')
axes[0].legend(fontsize=7, loc='upper right')
axes[0].grid(True, alpha=0.3)

# CLIP
for folder in sorted(unique_folders):
    mask = [i for i, m in enumerate(clip_metadata) if m['folder'] == folder]
    axes[1].scatter(clip_2d[mask, 0], clip_2d[mask, 1],
                    label=f'{folder} (n={len(mask)})',
                    color=folder_color_map[folder],
                    alpha=0.55, s=18, edgecolors='none')
axes[1].set_title(f'CLIP {CLIP_MODEL}\nPCA: {variance_explained[0]*100:.1f}% + {variance_explained[1]*100:.1f}% = {sum(variance_explained)*100:.1f}%',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel(f'PC1 ({variance_explained[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({variance_explained[1]*100:.1f}%)')
axes[1].legend(fontsize=8, loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.suptitle('PCA 2D: InsightFace (Per-wajah) vs CLIP (Per-foto)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 9. Simpulan

### Hasil Ekstraksi CLIP

| Metrik | Nilai |
|---|---|
| Model | ViT-L/14 |
| Total embedding | 2.533 (1 per foto) |
| Dimensi | 768 |
| Coverage | 100% foto |
| Normalisasi | L2 (cosine-ready) |

### Perbandingan Pendekatan

- **InsightFace** menghasilkan embedding **per wajah** → cocok untuk clustering identitas orang
- **CLIP** menghasilkan embedding **per foto** → cocok untuk clustering berdasarkan konten (scene, aktivitas, lokasi)
- CLIP memproses **semua 2.533 foto** termasuk 168 foto yang tidak ada wajahnya (diabaikan InsightFace)
- Kedua embedding disimpan di folder yang sama untuk perbandingan mudah pada notebook selanjutnya

### Next Step
Notebook **16_CLIP_HDBSCAN_Clustering.ipynb** — jalankan HDBSCAN pada CLIP embeddings dan bandingkan metrik clustering dengan hasil InsightFace.

In [ ]:
# Verifikasi file tersimpan
if os.path.exists(CLIP_EMB_PATH):
    size_mb = os.path.getsize(CLIP_EMB_PATH) / (1024 ** 2)
    print(f'File tersimpan: {CLIP_EMB_PATH}')
    print(f'  Ukuran file : {size_mb:.1f} MB')
    print(f'  Shape       : {clip_embeddings.shape}')
    print(f'\nPath untuk NB16:')
    print(f'  CLIP_EMB_PATH = "{CLIP_EMB_PATH}"')
    print(f'  OLD_EMB_PATH  = "{OLD_EMB_PATH}"')
else:
    print('WARNING: File belum tersimpan, jalankan ulang cell ekstraksi!')